In [1]:
%pip install langchain langchain-upstage pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 4.4 MB/s eta 0:00:00
  Using cached langchain_upstage-0.7.6-py3-none-any.whl (26 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.8/495.8 kB 18.1 MB/s eta 0:00:00
  Using cached langgraph-1.0.7-py3-none-any.whl (157 kB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached pypdf-6.6.2-py3-none-any.whl (329 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 30.9 MB/s eta 0:00:0000:0100:01
  Using cached langchain_openai-1.1.7-py3-none-any.whl (84 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.9 MB/s eta 0:00:00a 0:00:01
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.0/174.0 kB 8.7 MB/s et

In [35]:
from pydantic import BaseModel
from typing import List

class UserState(BaseModel):
    main_category: str      # 예: 학업, 인간관계, 진로
    sub_category: str       # 예: 학원 등록 고민, 시험 불안
    emotion: List[str]      # 예: 불안, 스트레스
    summary: str            # 고민 요약 (1문장)

In [1]:
example_state = UserState(
    main_category="학업",
    sub_category="학원 등록 여부 고민",
    emotion=["불안"],
    summary="성적 변동으로 인해 학원을 다녀야 할지 고민함"
)

example_state

NameError: name 'UserState' is not defined

## json 형식으로 사용자고민 듣고 임베딩하기 위해 필요한 json 생성.

In [44]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(model="solar-1-mini-chat")

In [45]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from fewshot import fewshot_example

# few-shot example prompt (아주 단순하게)
example_prompt = ChatPromptTemplate.from_template(
    "사용자 메시지:\n{input}\n\n출력:\n{output}"
)

few_shot_template = FewShotChatMessagePromptTemplate(
    examples=fewshot_example,
    example_prompt=example_prompt,
)


final_prompt = ChatPromptTemplate.from_messages([
    ("system", """
너는 사용자 메시지를 분석하는 AI다.
먼저 메시지가 분석 가능한지 판단하라.

[분석 불가능한 경우]
- 의미 없는 말
- 인사만 있는 경우
- 맥락이 없는 말

출력 형식:
{{
  "analysis_possible": false,
  "retry_message": "사용자에게 다시 질문하는 문장"
}}

[분석 가능한 경우]
{{
  "analysis_possible": true,
  "main_category": "",
  "sub_category": "",
  "emotion": [],
  "summary": "",
  "embedding_text": ""
}}

규칙:
- 모든 값은 한국어로 작성하라
- emotion은 최소 1개 이상 포함하라
- embedding_text는 유사도 계산용 자연어 문장이다
- JSON만 출력하라
"""),
    few_shot_template,
    ("user", "{input}")
])


ModuleNotFoundError: No module named 'fewshot'

In [39]:
analysis_chain = final_prompt | llm

response = analysis_chain.invoke({
    "input": "나 요즘 돈이 너무 부족해서 먹고살기 힘들다"
})

print(response.content)

{
  "analysis_possible": true,
  "main_category": "생활",
  "sub_category": "경제적 어려움",
  "emotion": ["고통", "불안"],
  "summary": "경제적인 문제로 일상생활에 어려움을 겪고 있음",
  "embedding_text": "사용자는 현재 금전적인 부족으로 인해 고통과 불안을 느끼고 있으며 먹고사는 데 어려움을 겪고 있다."
}


In [3]:
from langchain_core.documents import Document
import json

docs = []

with open(
    "/Users/kshi3430/jichini/jichini-ai/winter_vacation_project/user_worry.jsonl",
    encoding="utf-8"
) as f:
    for line in f:
        line = line.strip()

        # 빈 줄 방지 (중요)
        if not line:
            continue

        row = json.loads(line)

        docs.append(
            Document(
                page_content=row["embedding_text"],
                metadata={
                    "user_id": row["user_id"],
                    "province": row.get("province", ""),
                    "city": row.get("city", "")
                }
            )
        )

In [4]:
print(docs[40].page_content)

print(docs[40].metadata)


사용자는 가족 내에서 자신의 존재감이 점점 사라지는 것 같아 외로움을 느끼고 있다.
{'user_id': 'riverp_k11', 'province': '전라북도', 'city': '전주시'}


## 비슷한 고민을 찾아 주는지 테스트

In [5]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [6]:
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

In [7]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone


index_name="jichini-real-index"
database = PineconeVectorStore.from_documents(docs, embeddings,index_name=index_name)

# # 임베딩안하고 이미 만들어진거 사용하기
# database = PineconeVectorStore(
#     index_name="jichini-index",
#     embedding=embeddings,
#     pinecone_api_key=os.environ["PINECONE_API_KEY"],
# )


/Users/kshi3430/.pyenv/versions/jichini-ai/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
user_province = "경상북도"
user_city = "안동시"

retriever = database.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {
            "province": user_province,
            "city": user_city
        }
    }
)

In [128]:
from langchain_core.prompts import ChatPromptTemplate,FewShotChatMessagePromptTemplate



In [129]:
from dotenv import load_dotenv

load_dotenv()

True

In [130]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(model="solar-1-mini-chat")

## 가까운 지역끼리 추천

In [132]:
retrieved_docs = "\n\n".join(
    f"- user_id: {doc.metadata.get('user_id', '')}\n"
    f"- 지역: {doc.metadata.get('province', '')} {doc.metadata.get('city', '')}\n"
    f"- 고민 내용: {doc.page_content}"
    for doc in docs
)

In [ ]:
query = "요즘 여자친구를 사귀고 싶은데 어떻게 해야 할지 모르겠어"
prompt = f"""
[역할]
너는 고민 기반 사용자 매칭 결과만 출력하는 AI다.

[중요 규칙]
- 절대 추천 방법, 절차, 알고리즘을 설명하지 마라
- 추론 과정, 생각, 이유를 쓰지 마라
- Context에 없는 정보는 절대 만들지 마라

[매칭 기준]
1. 고민 내용 유사도
2. 지역 우선순위
   - 같은 시/군: 최우선
   - 같은 도/광역시: 다음 우선

[Context]
아래는 후보 사용자 목록이다.
각 항목은 다음 형식을 따른다.
- page_content: 고민 내용
- metadata.user_id
- metadata.province
- metadata.city

{retriever}

[사용자 고민]
{query}

[출력 형식]
- user_id:
- province:
- city:
- 유사한 고민 내용:

[출력 조건]
- 고민의 핵심 문제가 사용자 고민과 명확히 유사한 경우에만 출력하라
- 감정 또는 일부 키워드만 겹치는 경우는 출력하지 마라
- 조건을 만족하는 사용자가 없다면 출력하지 마라

위 형식만 사용해서 가장 적합한 사용자 최대 3명만 출력하라.
설명 문장은 절대 쓰지 마라.
"""


In [140]:
ai_message = llm.invoke(prompt)

In [141]:
ai_message.content

'- user_id: 12345\n- province: 경상북도\n- city: 안동시\n- 유사한 고민 내용: 최근에 이별하고 여자친구 사귀는 게 어렵다고 느껴요\n\n- user_id: 67890\n- province: 경상북도\n- city: 안동시\n- 유사한 고민 내용: 여자친구 사귀는 게 너무 어려워요. 어떻게 시작해야 할지 모르겠어요\n\n- user_id: 112233\n- province: 경상북도\n- city: 안동시\n- 유사한 고민 내용: 여자친구를 사귀고 싶은데, 어디서부터 시작해야 할지 모르겠어요'